# Bullwhip + incoherence example (matched)

This version is aligned with the standalone `bullwhip_CV_incoherence` and `incoherence` notebooks, but uses more robust file loading.

In [1]:

import os
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm


def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("None of these files exists:\n" + "\n".join(map(str, paths)))


def build_candidates(*stems, prefixes=("721", ""), extra_dirs=(".",)):
    paths = []
    for stem in stems:
        for d in extra_dirs:
            for prefix in prefixes:
                if prefix:
                    paths.append(os.path.join(d, f"{prefix}{stem}"))
                paths.append(os.path.join(d, stem))
    return list(dict.fromkeys(paths))


In [2]:

# ---------- configuration ----------
MODELS_DIR = r"E:/yjz/Extension for hts/JayCode/Models/"
PREFIXES = ("721", "")
LEAD_TIME = 1

result_dirs = (".", MODELS_DIR)

lgb_path = first_existing(build_candidates(f"lgbInvtSim_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
ets_path = first_existing(build_candidates(f"etsInvtSim_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
bu_path  = first_existing(build_candidates(f"BUOrder_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
tdfp_path = first_existing(build_candidates(f"TDFPOrder_L{LEAD_TIME}.pkl", prefixes=PREFIXES, extra_dirs=result_dirs))
mint_path = first_existing(build_candidates(
    f"VarOrder_L{LEAD_TIME}.pkl",
    f"MinTOrder_shrink_L{LEAD_TIME}.pkl",
    prefixes=PREFIXES,
    extra_dirs=result_dirs,
))

future_path = first_existing(build_candidates("future_28.pkl", prefixes=PREFIXES, extra_dirs=(MODELS_DIR, ".")))
tag_path = first_existing(build_candidates("tags.bin", "tags.pkl", prefixes=("",), extra_dirs=(MODELS_DIR, ".")))
actual_order_path = first_existing(["actual_order.pkl"])

lgb = pd.read_pickle(lgb_path)
ets = pd.read_pickle(ets_path)
base = pd.concat([lgb, ets], ignore_index=True)
bu = pd.read_pickle(bu_path)
tdfp = pd.read_pickle(tdfp_path)
mint = pd.read_pickle(mint_path)

uid_all = pd.read_pickle(future_path).reset_index(drop=True)
uid = uid_all[["unique_id", "ds"]]
tag = pd.read_pickle(tag_path)
ao = pd.read_pickle(actual_order_path)

print("Loaded:")
for x in [lgb_path, ets_path, bu_path, tdfp_path, mint_path, future_path, tag_path, actual_order_path]:
    print(" -", x)


Loaded:
 - .\lgbInvtSim_L1.pkl
 - .\etsInvtSim_L1.pkl
 - .\BUOrder_L1.pkl
 - .\TDFPOrder_L1.pkl
 - .\VarOrder_L1.pkl
 - E:/yjz/Extension for hts/JayCode/Models/721future_28.pkl
 - E:/yjz/Extension for hts/JayCode/Models/tags.bin
 - actual_order.pkl


## orders_collector

In [3]:

orders = base[["name", "true_demand", "forecasts"]].copy()
for tsl in tqdm(["90", "95", "99"]):
    orders[f"ot_{tsl}"] = base[f"ot_{tsl}"].to_numpy()
    orders[f"ot_bu{tsl}"] = bu[f"ot_{tsl}"].to_numpy()
    orders[f"ot_tdfp{tsl}"] = tdfp[f"ot_{tsl}"].to_numpy()
    orders[f"ot_mint{tsl}"] = mint[f"ot_{tsl}"].to_numpy()

orders.to_pickle("all_order.pkl")
print("saved:", os.path.abspath("all_order.pkl"))
orders.head()


100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 10.57it/s]


saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\all_order.pkl


,name,true_demand,forecasts,ot_90,ot_bu90,ot_tdfp90,ot_mint90,ot_95,ot_bu95,ot_tdfp95,ot_mint95,ot_99,ot_bu99,ot_tdfp99,ot_mint99
0,lgb_base,4.0,5.689794,2.989336,4.350416,2.989336,3.558033,2.989336,4.350416,2.989336,3.558033,2.989336,4.350416,2.989336,3.558033
1,lgb_base,5.0,4.679130,5.119798,4.451770,5.119798,4.807542,5.119798,4.451770,5.119798,4.807542,5.119798,4.451770,5.119798,4.807542
2,lgb_base,7.0,4.798928,8.181289,7.910317,8.181289,7.931229,8.181289,7.910317,8.181289,7.931229,8.181289,7.910317,8.181289,7.931229
3,lgb_base,1.0,5.980217,0.068346,1.017425,0.068346,0.783449,0.068346,1.017425,0.068346,0.783449,0.068346,1.017425,0.068346,0.783449
4,lgb_base,9.0,5.048564,12.199352,11.998162,12.199352,11.936915,12.199352,11.998162,12.199352,11.936915,12.199352,11.998162,12.199352,11.936915


In [25]:
a1 = orders['ot_90'].round(3)
a2 = orders['ot_95'].round(3)
a3 = orders['ot_99'].round(3)

In [24]:
a1[0]

2.989

In [22]:
a2[0]#.values

2.989336044955536

## products_list

In [4]:

if isinstance(tag, dict):
    if "total/cat_id/dept_id/item_id" in tag:
        series_ids = list(tag["total/cat_id/dept_id/item_id"])
    else:
        raise KeyError("Cannot find key 'total/cat_id/dept_id/item_id' in tag dictionary.")
else:
    # fallback for DataFrame/Series-like tag files
    if "total/cat_id/dept_id/item_id" in getattr(tag, "columns", []):
        series_ids = tag["total/cat_id/dept_id/item_id"].tolist()
    else:
        raise KeyError("Cannot find 'total/cat_id/dept_id/item_id' in tag object.")

prdd = pd.Series(series_ids, name="products").astype(str).str.split("/")
prd = [parts[-1] for parts in prdd]
prd_idx = pd.DataFrame({
    p: uid_all[uid_all["unique_id"].astype(str).str.contains(fr"{p}$", regex=True)].index.tolist()
    for p in tqdm(prd)
})
prd_idx.to_pickle("prd_idx.pkl")
print("saved:", os.path.abspath("prd_idx.pkl"))
prd_idx.iloc[:5, :5]


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [24:57<00:00,  2.04it/s]


saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\prd_idx.pkl


,FOODS_1_001,FOODS_1_002,FOODS_1_003,FOODS_1_004,FOODS_1_005
0,0,28,56,84,112
1,1,29,57,85,113
2,2,30,58,86,114
3,3,31,59,87,115
4,4,32,60,88,116


## CV and Bullwhip Effect Ratio

In [14]:
orders = pd.read_pickle("all_order.pkl")
uid = uid_all[["unique_id", "ds"]].copy()

lst = orders["name"].unique().tolist()
frdr_ = pd.concat(
    [pd.concat([uid.reset_index(drop=True), orders[orders["name"] == name].reset_index(drop=True)], axis=1) for name in tqdm(lst)],
    ignore_index=True,
)

dff = frdr_.copy()
split_cols = dff["unique_id"].astype(str).str.split("/")
dff["product"] = [parts[-1] for parts in split_cols]
dff["level"] = [len(parts) - 3 for parts in split_cols]
dff.head()


100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:07<00:00,  1.05it/s]


,unique_id,ds,name,true_demand,forecasts,ot_90,ot_bu90,ot_tdfp90,ot_mint90,ot_95,ot_bu95,ot_tdfp95,ot_mint95,ot_99,ot_bu99,ot_tdfp99,ot_mint99,product,level
0,TOTAL/FOODS/FOODS_1/FOODS_1_001,1914,lgb_base,4.0,5.689794,2.989336,4.350416,2.989336,3.558033,2.989336,4.350416,2.989336,3.558033,2.989336,4.350416,2.989336,3.558033,FOODS_1_001,1
1,TOTAL/FOODS/FOODS_1/FOODS_1_001,1915,lgb_base,5.0,4.679130,5.119798,4.451770,5.119798,4.807542,5.119798,4.451770,5.119798,4.807542,5.119798,4.451770,5.119798,4.807542,FOODS_1_001,1
2,TOTAL/FOODS/FOODS_1/FOODS_1_001,1916,lgb_base,7.0,4.798928,8.181289,7.910317,8.181289,7.931229,8.181289,7.910317,8.181289,7.931229,8.181289,7.910317,8.181289,7.931229,FOODS_1_001,1
3,TOTAL/FOODS/FOODS_1/FOODS_1_001,1917,lgb_base,1.0,5.980217,0.068346,1.017425,0.068346,0.783449,0.068346,1.017425,0.068346,0.783449,0.068346,1.017425,0.068346,0.783449,FOODS_1_001,1
4,TOTAL/FOODS/FOODS_1/FOODS_1_001,1918,lgb_base,9.0,5.048564,12.199352,11.998162,12.199352,11.936915,12.199352,11.998162,12.199352,11.936915,12.199352,11.998162,12.199352,11.936915,FOODS_1_001,1


In [ ]:
dff

In [6]:
var_ = dff.groupby(["unique_id", "name", "product", "level"]).var(numeric_only=True).drop(columns=["forecasts"], errors="ignore")
metric_cols = [c for c in var_.columns if c != "true_demand"]
var_[metric_cols] = var_[metric_cols].div(var_["true_demand"], axis=0)
var_.head()


ds  \
unique_id                               name     product     level              
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      35.062371   
                                        ets_bu   FOODS_1_001 3      35.062371   
                                        ets_mint FOODS_1_001 3      35.062371   
                                        ets_td   FOODS_1_001 3      35.062371   
                                        lgb_base FOODS_1_001 3      35.062371   

                                                                    true_demand  \
unique_id                               name     product     level                
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3         1.929894   
                                        ets_bu   FOODS_1_001 3         1.929894   
                                        ets_mint FOODS_1_001 3         1.929894   
                                        ets_td   FOODS_1_001 3         1.929894   
                                        lgb_base FOODS_1_001 3         1.929894   

                                                                       ot_90  \
unique_id                               name     product     level             
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      1.038273   
                                        ets_bu   FOODS_1_001 3      1.038273   
                                        ets_mint FOODS_1_001 3      1.025573   
                                        ets_td   FOODS_1_001 3      1.034815   
                                        lgb_base FOODS_1_001 3      1.271940   

                                                                     ot_bu90  \
unique_id                               name     product     level             
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      1.038273   
                                        ets_bu   FOODS_1_001 3      1.038273   
                                        ets_mint FOODS_1_001 3      1.025573   
                                        ets_td   FOODS_1_001 3      1.034815   
                                        lgb_base FOODS_1_001 3      1.271940   

                                                                    ot_tdfp90  \
unique_id                               name     product     level              
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3       1.079301   
                                        ets_bu   FOODS_1_001 3       1.040926   
                                        ets_mint FOODS_1_001 3       1.158745   
                                        ets_td   FOODS_1_001 3       1.077822   
                                        lgb_base FOODS_1_001 3       1.512728   

                                                                    ot_mint90  \
unique_id                               name     product     level              
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3       1.038599   
                                        ets_bu   FOODS_1_001 3       1.037236   
                                        ets_mint FOODS_1_001 3       1.029431   
                                        ets_td   FOODS_1_001 3       1.035644   
                                        lgb_base FOODS_1_001 3       1.277966   

                                                                       ot_95  \
unique_id                               name     product     level             
TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001 ets_base FOODS_1_001 3      1.038273   
                                        ets_bu   FOODS_1_001 3      1.038273   
                                        ets_mint FOODS_1_001 3      1.025573   
                                        ets_td   FOODS_1_001 3      1.034815   
                                        lgb_base FOODS_1_001 3      1.271940   

                                                                     ot_bu95  \
unique_id                 

In [13]:
bwe_summary.iloc[:, 2:]

ot_90   ot_bu90  ot_tdfp90  ot_mint90     ot_95   ot_bu95  \
level name                                                                     
1     ets_base  1.373366  1.116341   1.373366   1.219460  1.373366  1.116341   
      ets_bu    1.212917  1.116341   1.212917   1.160778  1.212917  1.116341   
      ets_mint  1.325085  1.182200   1.325085   1.245004  1.325085  1.182200   
      ets_td    1.373366  1.227162   1.373366   1.293861  1.373366  1.227162   
      lgb_base  1.667579  1.330667   1.667579   1.457954  1.667579  1.330667   
      lgb_bu    1.573816  1.330667   1.573816   1.429194  1.573816  1.330667   
      lgb_mint  1.626448  1.376563   1.626448   1.480260  1.626448  1.376563   
      lgb_td    1.667579  1.392066   1.667579   1.507054  1.667579  1.392066   
2     ets_base  1.179900  1.087088   1.227702   1.141638  1.179900  1.087088   
      ets_bu    1.125636  1.087088   1.129005   1.106982  1.125636  1.087088   
      ets_mint  1.164671  1.109867   1.174302   1.137070  1.164671  1.109867   
      ets_td    1.214602  1.157228   1.224534   1.187100  1.214602  1.157228   
      lgb_base  1.435875  1.293310   1.492097   1.359876  1.435875  1.293310   
      lgb_bu    1.400057  1.293310   1.426805   1.338775  1.400057  1.293310   
      lgb_mint  1.419176  1.308261   1.443638   1.357234  1.419176  1.308261   
      lgb_td    1.457288  1.332779   1.487334   1.389318  1.457288  1.332779   
3     ets_base  1.065013  1.065013   1.099625   1.072737  1.065013  1.065013   
      ets_bu    1.065013  1.065013   1.057337   1.063498  1.065013  1.065013   
      ets_mint  1.055627  1.055627   1.060592   1.054065  1.055627  1.055627   
      ets_td    1.090694  1.090694   1.094137   1.090904  1.090694  1.090694   
      lgb_base  1.262423  1.262423   1.301042   1.261106  1.262423  1.262423   
      lgb_bu    1.262423  1.262423   1.276882   1.254765  1.262423  1.262423   
      lgb_mint  1.262837  1.262837   1.271818   1.256937  1.262837  1.262837   
      lgb_td    1.265139  1.265139   1.290771   1.261610  1.265139  1.265139   

                ot_tdfp95  ot_mint95     ot_99   ot_bu99  ot_tdfp99  ot_mint99  
level name                                                                      
1     ets_base   1.373366   1.219460  1.373366  1.116341   1.373366   1.219460  
      ets_bu     1.212917   1.160778  1.212917  1.116341   1.212917   1.160778  
      ets_mint   1.325085   1.245004  1.325085  1.182200   1.325085   1.245004  
      ets_td     1.373366   1.293861  1.373366  1.227162   1.373366   1.293861  
      lgb_base   1.667579   1.457954  1.667579  1.330667   1.667579   1.457954  
      lgb_bu     1.573816   1.429194  1.573816  1.330667   1.573816   1.429194  
      lgb_mint   1.626448   1.480260  1.626448  1.376563   1.626448   1.480260  
      lgb_td     1.667579   1.507054  1.667579  1.392066   1.667579   1.507054  
2     ets_base   1.227702   1.141638  1.179900  1.087088   1.227702   1.141638  
      ets_bu     1.129005   1.106982  1.125636  1.087088   1.129005   1.106982  
      ets_mint   1.174302   1.137070  1.164671  1.109867   1.174302   1.137070  
      ets_td     1.224534   1.187100  1.214602  1.157228   1.224534   1.187100  
      lgb_base   1.492097   1.359876  1.435875  1.293310   1.492097   1.359876  
      lgb_bu     1.426805   1.338775  1.400057  1.293310   1.426805   1.338775  
      lgb_mint   1.443638   1.357234  1.419176  1.308261   1.443638   1.357234  
      lgb_td     1.487334   1.389318  1.457288  1.332779   1.487334   1.389318  
3     ets_base   1.099625   1.072737  1.065013  1.065013   1.099625   1.072737  
      ets_bu     1.057337   1.063498  1.065013  1.065013   1.057337   1.063498  
      ets_mint   1.060592   1.054065  1.055627  1.055627   1.060592   1.054065  
      ets_td     1.094137   1.090904  1.090694  1.090694   1.094137   1.090904  
      lgb_base   1.301042   1.261106  1.262423  1.262423   1.301042   1.261106  
      lgb_bu     1.276882   1.254765  1.262423  1.262423   1.276882   1.254

In [7]:

df2 = var_[np.isfinite(var_).all(axis=1)].copy()

vlist = []
product_ids = df2.index.get_level_values("product").unique().tolist()
for product_id in tqdm(product_ids):
    one = df2.xs(product_id, level="product").groupby(["level", "name"]).mean(numeric_only=True)
    vlist.append(one)

bwe_summary = pd.concat(vlist).groupby(["level", "name"]).mean(numeric_only=True)
median_bwe = df2.reset_index().drop(columns=["unique_id", "product"]).groupby(["name", "level"]).median(numeric_only=True).drop(columns=["true_demand"], errors="ignore")

bwe_summary.iloc[:, 3:].to_csv("bwe.csv")
median_bwe.reset_index().to_csv("median_bwe.csv", index=False)

print("saved:", os.path.abspath("bwe.csv"))
print("saved:", os.path.abspath("median_bwe.csv"))
bwe_summary.iloc[:, 3:]


100%|█████████████████████████████████████████████████████████████████████████████| 3042/3042 [00:06<00:00, 492.31it/s]


saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\bwe.csv
saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\median_bwe.csv


ot_bu90  ot_tdfp90  ot_mint90     ot_95   ot_bu95  ot_tdfp95  \
level name                                                                      
1     ets_base  1.116341   1.373366   1.219460  1.373366  1.116341   1.373366   
      ets_bu    1.116341   1.212917   1.160778  1.212917  1.116341   1.212917   
      ets_mint  1.182200   1.325085   1.245004  1.325085  1.182200   1.325085   
      ets_td    1.227162   1.373366   1.293861  1.373366  1.227162   1.373366   
      lgb_base  1.330667   1.667579   1.457954  1.667579  1.330667   1.667579   
      lgb_bu    1.330667   1.573816   1.429194  1.573816  1.330667   1.573816   
      lgb_mint  1.376563   1.626448   1.480260  1.626448  1.376563   1.626448   
      lgb_td    1.392066   1.667579   1.507054  1.667579  1.392066   1.667579   
2     ets_base  1.087088   1.227702   1.141638  1.179900  1.087088   1.227702   
      ets_bu    1.087088   1.129005   1.106982  1.125636  1.087088   1.129005   
      ets_mint  1.109867   1.174302   1.137070  1.164671  1.109867   1.174302   
      ets_td    1.157228   1.224534   1.187100  1.214602  1.157228   1.224534   
      lgb_base  1.293310   1.492097   1.359876  1.435875  1.293310   1.492097   
      lgb_bu    1.293310   1.426805   1.338775  1.400057  1.293310   1.426805   
      lgb_mint  1.308261   1.443638   1.357234  1.419176  1.308261   1.443638   
      lgb_td    1.332779   1.487334   1.389318  1.457288  1.332779   1.487334   
3     ets_base  1.065013   1.099625   1.072737  1.065013  1.065013   1.099625   
      ets_bu    1.065013   1.057337   1.063498  1.065013  1.065013   1.057337   
      ets_mint  1.055627   1.060592   1.054065  1.055627  1.055627   1.060592   
      ets_td    1.090694   1.094137   1.090904  1.090694  1.090694   1.094137   
      lgb_base  1.262423   1.301042   1.261106  1.262423  1.262423   1.301042   
      lgb_bu    1.262423   1.276882   1.254765  1.262423  1.262423   1.276882   
      lgb_mint  1.262837   1.271818   1.256937  1.262837  1.262837   1.271818   
      lgb_td    1.265139   1.290771   1.261610  1.265139  1.265139   1.290771   

                ot_mint95     ot_99   ot_bu99  ot_tdfp99  ot_mint99  
level name                                                           
1     ets_base   1.219460  1.373366  1.116341   1.373366   1.219460  
      ets_bu     1.160778  1.212917  1.116341   1.212917   1.160778  
      ets_mint   1.245004  1.325085  1.182200   1.325085   1.245004  
      ets_td     1.293861  1.373366  1.227162   1.373366   1.293861  
      lgb_base   1.457954  1.667579  1.330667   1.667579   1.457954  
      lgb_bu     1.429194  1.573816  1.330667   1.573816   1.429194  
      lgb_mint   1.480260  1.626448  1.376563   1.626448   1.480260  
      lgb_td     1.507054  1.667579  1.392066   1.667579   1.507054  
2     ets_base   1.141638  1.179900  1.087088   1.227702   1.141638  
      ets_bu     1.106982  1.125636  1.087088   1.129005   1.106982  
      ets_mint   1.137070  1.164671  1.109867   1.174302   1.137070  
      ets_td     1.187100  1.214602  1.157228   1.224534   1.187100  
      lgb_base   1.359876  1.435875  1.293310   1.492097   1.359876  
      lgb_bu     1.338775  1.400057  1.293310   1.426805   1.338775  
      lgb_mint   1.357234  1.419176  1.308261   1.443638   1.357234  
      lgb_td     1.389318  1.457288  1.332779   1.487334   1.389318  
3     ets_base   1.072737  1.065013  1.065013   1.099625   1.072737  
      ets_bu     1.063498  1.065013  1.065013   1.057337   1.063498  
      ets_mint   1.054065  1.055627  1.055627   1.060592   1.054065  
      ets_td     1.090904  1.090694  1.090694   1.094137   1.090904  
      lgb_base   1.261106  1.262423  1.262423   1.301042   1.261106  
      lgb_bu     1.254765  1.262423  1.262423   1.276882   1.254765  
      lgb_mint   1.256937  1.262837  1.262837   1.271818   1.256937  
      lgb_td     1.261610  1.265139  1.265139   1.290771   1.261610

## Incoherence

In [8]:

lgb_methods = list(lgb["name"].unique())
ets_methods = list(ets["name"].unique())

df_lgb = lgb[["name", "true_demand", "ot_90", "ot_95", "ot_99"]].copy()
df_ets = ets[["name", "true_demand", "ot_90", "ot_95", "ot_99"]].copy()


methods = ["ot_90", "ot_95", "ot_99"]
prd_list = list(prd_idx.columns)


In [9]:

from typing import List


def get_direct_children(all_ids: List[str], parent_id: str) -> List[str]:
    parts = str(parent_id).split("/")
    depth = len(parts)
    children = []
    for sid in all_ids:
        sid = str(sid)
        if sid == parent_id:
            continue
        child_parts = sid.split("/")
        if len(child_parts) != depth + 1:
            continue
        if (
            child_parts[0] == parts[0]
            and child_parts[-3:] == parts[-3:]
            and child_parts[1:-3][: len(parts[1:-3])] == parts[1:-3]
        ):
            children.append(sid)
    return sorted(children)


def check_coherence_long_format(df: pd.DataFrame, value_col: str, id_col: str = "unique_id", time_col: str = "ds") -> pd.DataFrame:
    all_ids = df[id_col].astype(str).unique().tolist()
    results = []
    for parent_id in all_ids:
        children = get_direct_children(all_ids, parent_id)
        if not children:
            continue

        parent_df = df[df[id_col].astype(str) == parent_id][[time_col, value_col]].sort_values(time_col)
        children_df = df[df[id_col].astype(str).isin(children)][[time_col, id_col, value_col]]
        children_sum = children_df.groupby(time_col)[value_col].sum().reset_index()

        merged = pd.merge(parent_df, children_sum, on=time_col, suffixes=("_parent", "_children"))
        merged["diff"] = merged[f"{value_col}_parent"] - merged[f"{value_col}_children"]
        merged["abs_diff"] = merged["diff"].abs()
        merged["parent_id"] = parent_id
        results.append(merged)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


def run_incoherence_for(df_source: pd.DataFrame, method_names: list):
    incoherence = {}
    for m in methods:
        m_inco = {}
        for fr in method_names:
            print(f"{fr} Begins", "*" * 70)
            df_r = df_source[df_source["name"] == fr].reset_index(drop=True)
            check_result = {}
            for prd in tqdm(prd_list):
                idx = prd_idx[prd].dropna().astype(int).tolist()
                if len(idx) == 0:
                    continue
                dff = pd.concat([uid.iloc[idx, :].reset_index(drop=True), df_r.iloc[idx, :].reset_index(drop=True)], axis=1)
                check_result[prd] = check_coherence_long_format(dff, m)
            m_inco[fr] = check_result
            print(f"{fr} Finished", "*" * 70)
        incoherence[m] = m_inco
    return incoherence


In [10]:

import pickle

lgb_results = run_incoherence_for(df_lgb, lgb_methods)
with open("lgb_incoherence_results.pkl", "wb") as f:
    pickle.dump(lgb_results, f)

ets_results = run_incoherence_for(df_ets, ets_methods)
with open("ets_incoherence_results.pkl", "wb") as f:
    pickle.dump(ets_results, f)

print("saved:", os.path.abspath("lgb_incoherence_results.pkl"))
print("saved:", os.path.abspath("ets_incoherence_results.pkl"))


lgb_base Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:23<00:00, 36.54it/s]


lgb_base Finished **********************************************************************
lgb_bu Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:23<00:00, 36.45it/s]


lgb_bu Finished **********************************************************************
lgb_td Begins **********************************************************************


...

ets_bu Finished **********************************************************************
ets_td Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:23<00:00, 36.65it/s]


ets_td Finished **********************************************************************
ets_mint Begins **********************************************************************


100%|██████████████████████████████████████████████████████████████████████████████| 3049/3049 [01:28<00:00, 34.48it/s]


ets_mint Finished **********************************************************************
saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\lgb_incoherence_results.pkl
saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\ets_incoherence_results.pkl


In [11]:

import pickle


def avg_incoherence(result_dict: dict):
    pieces = []
    for key_1 in list(result_dict.keys()):
        dict_2 = result_dict[key_1]
        dict_4 = {}
        for key_2 in tqdm(list(dict_2.keys())):
            one = dict_2[key_2]
            if len(one) == 0:
                continue
            prodcs = one.groupby("parent_id").mean(numeric_only=True)
            dict_4[key_2] = prodcs["abs_diff"] / prodcs.iloc[:, 1]
        pieces.append(pd.DataFrame({key_1: dict_4}))
    return pd.concat(pieces, axis=1) if pieces else pd.DataFrame()


lgb_incoherence = {i: avg_incoherence(lgb_results[i]) for i in list(lgb_results.keys())}
ets_incoherence = {i: avg_incoherence(ets_results[i]) for i in list(ets_results.keys())}

a = {}
for i in methods:
    a[i] = pd.concat([lgb_incoherence[i], ets_incoherence[i]], axis=1)

out_ = {}
for i in methods:
    stacked = a[i].stack(dropna=True)
    parts = []
    for (product, method), s in tqdm(stacked.items()):
        if isinstance(s, pd.Series) and len(s) > 0:
            tmp = s.rename_axis("node_name").reset_index(name="incoherence_ratio")
            tmp["product"] = product
            tmp["method"] = method
            depth = tmp["node_name"].astype(str).str.count("/")
            tmp["level"] = (depth - depth.min() + 1).astype(int)
            parts.append(tmp[["method", "product", "node_name", "level", "incoherence_ratio"]])
    out_[i] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["method", "product", "node_name", "level", "incoherence_ratio"])

b = {}
for i in methods:
    b[i] = out_[i][["method", "level", "incoherence_ratio"]].groupby(["method", "level"]).mean(numeric_only=True)
    b[i] = b[i].rename(columns={"incoherence_ratio": f"{i}_incoherence_ratio"})

results = pd.merge(b["ot_90"], b["ot_95"], on=["method", "level"], how="outer")
results = pd.merge(results, b["ot_99"], on=["method", "level"], how="outer")
results.to_csv("incoherence_ratio_for_all_tsl.csv")

print("saved:", os.path.abspath("incoherence_ratio_for_all_tsl.csv"))
results


100%|█████████████████████████████████████████████████████████████████████████████| 3049/3049 [00:04<00:00, 688.41it/s]
C:\Users\qifei\AppData\Local\Temp\ipykernel_39560\1712161910.py:28: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = a[i].stack(dropna=True)
24392it [01:12, 336.45it/s]
C:\Users\qifei\AppData\Local\Temp\ipykernel_39560\1712161910.py:28: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = a[i].stack(dropna=True)
24392it [01:12, 337.60it/s]
C:\Users\qifei\AppData\Local\Temp\ipykernel_39560\1712161910.py:28: FutureWarning: The previous implemen

saved: E:\yjz\Extension for hts\JayCode\Modified_Sim2\incoherence_ratio_for_all_tsl.csv


ot_90_incoherence_ratio  ot_95_incoherence_ratio  \
method   level                                                     
ets_base 1                     0.037066                 0.037066   
         2                          inf                      inf   
ets_bu   1                          inf                      inf   
         2                          inf                      inf   
ets_mint 1                     0.025764                 0.025764   
         2                          inf                      inf   
ets_td   1                     0.025720                 0.025720   
         2                          inf                      inf   
lgb_base 1                     0.092745                 0.092745   
         2                          inf                      inf   
lgb_bu   1                     0.061761                 0.061761   
         2                     0.117239                 0.117239   
lgb_mint 1                     0.060816                 0.060816   
         2                          inf                      inf   
lgb_td   1                     0.061824                 0.061824   
         2                     0.108496                 0.108496   

                ot_99_incoherence_ratio  
method   level                           
ets_base 1                     0.037066  
         2                          inf  
ets_bu   1                          inf  
         2                          inf  
ets_mint 1                     0.025764  
         2                          inf  
ets_td   1                     0.025720  
         2                          inf  
lgb_base 1                     0.092745  
         2                          inf  
lgb_bu   1                     0.061761  
         2                     0.117239  
lgb_mint 1                     0.060816  
         2                          inf  
lgb_td   1                     0.061824  
         2                     0.108496

In [12]:
results

ot_90_incoherence_ratio  ot_95_incoherence_ratio  \
method   level                                                     
ets_base 1                     0.037066                 0.037066   
         2                          inf                      inf   
ets_bu   1                          inf                      inf   
         2                          inf                      inf   
ets_mint 1                     0.025764                 0.025764   
         2                          inf                      inf   
ets_td   1                     0.025720                 0.025720   
         2                          inf                      inf   
lgb_base 1                     0.092745                 0.092745   
         2                          inf                      inf   
lgb_bu   1                     0.061761                 0.061761   
         2                     0.117239                 0.117239   
lgb_mint 1                     0.060816                 0.060816   
         2                          inf                      inf   
lgb_td   1                     0.061824                 0.061824   
         2                     0.108496                 0.108496   

                ot_99_incoherence_ratio  
method   level                           
ets_base 1                     0.037066  
         2                          inf  
ets_bu   1                          inf  
         2                          inf  
ets_mint 1                     0.025764  
         2                          inf  
ets_td   1                     0.025720  
         2                          inf  
lgb_base 1                     0.092745  
         2                          inf  
lgb_bu   1                     0.061761  
         2                     0.117239  
lgb_mint 1                     0.060816  
         2                          inf  
lgb_td   1                     0.061824  
         2                     0.108496